In [ ]:
"""
Quantum Difference Interference Filter — 1D Prototype
======================================================
Goal: Prepare a state where only boundary anchors between
valid (all-zero) and non-valid windows survive via destructive
interference, WITHOUT Grover or post-selection as the core idea.

Core idea:
  For each anchor a, define g(a) = 1 iff window w_a = 0^M.
  Prepare: sum_a |a> (|g(a)> - |g(a+1)>)
  Interior of constant g regions cancel; only g(a) != g(a+1) survive.

This is O(1) oracle calls per anchor — the superposition is prepared
in one pass (no amplitude amplification), so the query complexity
matches a single Grover oracle call, not O(sqrt(N)).
The "free" quantum speedup comes from the interference cancellation
doing the filtering work during preparation itself.
"""

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import XGate
import itertools


# ─────────────────────────────────────────────────────────────────────────────
# Classical helpers
# ─────────────────────────────────────────────────────────────────────────────

def compute_g(occ: list[int], M: int) -> list[int]:
    """
    g(a) = 1 iff occ[a:a+M] == [0]*M  (fully empty window).
    Returns list of length W = N - M + 1.
    """
    N = len(occ)
    W = N - M + 1
    return [int(all(occ[a + k] == 0 for k in range(M))) for a in range(W)]


def num_qubits_for(n: int) -> int:
    """Minimum qubits to represent integers 0..n-1."""
    return max(1, int(np.ceil(np.log2(max(n, 2)))))


# ─────────────────────────────────────────────────────────────────────────────
# Oracle: controlled-g(a) or controlled-g(a+1)
# ─────────────────────────────────────────────────────────────────────────────

def build_g_oracle(g_values: list[int], offset: int, n_a: int) -> QuantumCircuit:
    """
    Build an oracle circuit that flips qubit g_q if g(a + offset) == 1,
    for each computational basis state |a> of the anchor register.

    Parameters
    ----------
    g_values : list of g(a) for a = 0..W-1
    offset   : 0 → compute g(a); 1 → compute g(a+1)
    n_a      : number of qubits in the anchor register

    Returns
    -------
    QuantumCircuit on n_a + 1 qubits  [a_{n_a-1}..a_0, g]
    """
    qc = QuantumCircuit(n_a + 1, name=f"Oracle_g(a+{offset})")
    g_q = n_a  # index of the g qubit in this sub-circuit

    W = len(g_values)
    # We iterate over anchors a = 0..W-2 (we only need W-1 differences)
    for a in range(W - 1):
        target = a + offset
        if target < 0 or target >= W:
            continue
        if g_values[target] == 1:
            # Multi-controlled X: flip g_q when anchor register == a
            # Encode a in binary; apply X gates to 0-bits, MCX, uncompute.
            bits = [(a >> bit) & 1 for bit in range(n_a)]
            # X on qubits where bit == 0 (to make |a> look like |1...1>)
            for bit_idx, b in enumerate(bits):
                if b == 0:
                    qc.x(bit_idx)
            # Multi-controlled Toffoli targeting g_q
            ctrl_qubits = list(range(n_a))
            qc.mcx(ctrl_qubits, g_q)
            # Uncompute
            for bit_idx, b in enumerate(bits):
                if b == 0:
                    qc.x(bit_idx)

    return qc


# ─────────────────────────────────────────────────────────────────────────────
# Uniform superposition over a = 0..W-2
# ─────────────────────────────────────────────────────────────────────────────

def uniform_superposition_over_range(n_a: int, W_minus_1: int) -> QuantumCircuit:
    """
    Prepare (1/sqrt(W-1)) * sum_{a=0}^{W-2} |a>.

    Strategy: use Qiskit's initialize() with an explicit amplitude vector.
    This is exact and avoids manual W-qubit arithmetic circuits.
    """
    dim = 2 ** n_a
    amps = np.zeros(dim, dtype=complex)
    for a in range(W_minus_1):
        amps[a] = 1.0 / np.sqrt(W_minus_1)

    qc = QuantumCircuit(n_a, name="UniformAnchor")
    qc.initialize(amps, list(range(n_a)))
    return qc


# ─────────────────────────────────────────────────────────────────────────────
# Main circuit builder
# ─────────────────────────────────────────────────────────────────────────────

def build_difference_circuit(occ: list[int], M: int) -> QuantumCircuit:
    """
    Build the full interference-difference circuit.

    THE KEY FIX: use TWO separate g qubits (g0, g1), one per branch.
    With a shared g qubit, oracle_1 would write on top of what oracle_0
    already wrote, causing both branches to corrupt each other — that
    was the bug in the previous version.

    Registers
    ---------
    a_reg  : anchor index, n_a qubits
    p_reg  : path ancilla, 1 qubit
    g0_reg : validity qubit for the p=0 branch  → stores g(a)
    g1_reg : validity qubit for the p=1 branch  → stores g(a+1)

    Circuit structure
    -----------------
    1. Prepare |a> in uniform superposition over a = 0..W-2.

    2. Prepare path ancilla in |-> = (|0> - |1>) / sqrt(2).
       This injects the minus sign between the two branches before
       any oracle fires — so both branches start with opposite global
       phases.

    3a. Controlled on p=0: compute g(a)   into g0_reg.
    3b. Controlled on p=1: compute g(a+1) into g1_reg.

       State before recombination:
         (1/√(W-1)) (1/√2) Σ_a |a⟩ (|0⟩_p |g(a)⟩_g0 |0⟩_g1
                                   − |1⟩_p |0⟩_g0 |g(a+1)⟩_g1)

    4. Apply H to path ancilla to recombine branches.

       After H on p, using |0⟩=(|+⟩+|−⟩)/√2, |1⟩=(|+⟩−|−⟩)/√2:

         p=0 output  ∝  |g(a)⟩_g0|0⟩_g1  −  |0⟩_g0|g(a+1)⟩_g1   ← DIFFERENCE ★
         p=1 output  ∝  |g(a)⟩_g0|0⟩_g1  +  |0⟩_g0|g(a+1)⟩_g1   ← sum, discard

       Wait — with the |−⟩ initialisation the useful branch is p=0.
       Let's verify: starting from |−⟩_p = (|0⟩−|1⟩)/√2,
         after conditional oracles:  (|0⟩|g(a)⟩_g0|0⟩_g1 − |1⟩|0⟩_g0|g(a+1)⟩_g1)/√2
         after H on p:
           |0⟩_p → contributes  (|g(a)⟩_g0|0⟩_g1 − |0⟩_g0|g(a+1)⟩_g1) / 2   ★
           |1⟩_p → contributes  (|g(a)⟩_g0|0⟩_g1 + |0⟩_g0|g(a+1)⟩_g1) / 2

       So the DIFFERENCE lives in the p=0 branch after recombination.

    Cancellation condition:
      In the p=0 branch, the amplitude for anchor a is proportional to
        |g(a)⟩_g0|0⟩_g1  −  |0⟩_g0|g(a+1)⟩_g1
      This is zero iff g(a) == g(a+1):
        • g(a)=g(a+1)=0: |0⟩|0⟩ − |0⟩|0⟩ = 0  ✓ cancels
        • g(a)=g(a+1)=1: |1⟩|0⟩ − |0⟩|1⟩ ≠ 0  ✗ does NOT cancel
      Wait — when g(a)=g(a+1)=1 the two terms are orthogonal vectors,
      so their *norm* is NOT zero. The cancellation only works when
      g(a)=g(a+1)=0.

      The correct interpretation:
        The two terms cancel (same vector with opposite sign) only when
        the g values are both 0 (both terms = |0⟩_g0|0⟩_g1 with opposite sign).
        When g(a)=g(a+1)=1: terms are |1⟩|0⟩ and |0⟩|1⟩ — orthogonal, no cancel.
        When g(a)≠g(a+1): one term is |0⟩|0⟩ and the other |1⟩|0⟩ or |0⟩|1⟩ — no cancel.

      To achieve cancellation for ALL constant regions (both 0-0 and 1-1),
      we need a single shared g qubit but must ensure both branches write
      to the SAME basis state when g(a)=g(a+1). The trick: instead of
      storing g values in separate qubits, compute the XOR g(a)⊕g(a+1)
      directly into a single ancilla, which is 0 iff g(a)==g(a+1).

    ── REVISED APPROACH ──────────────────────────────────────────────────
    Use a single ancilla qubit `d` (for "difference/XOR"):
      • Controlled on p=0: flip d if g(a)=1      (i.e., add g(a) mod 2)
      • Controlled on p=1: flip d if g(a+1)=1    (i.e., add g(a+1) mod 2)
    But this still mixes branches — p is in superposition when oracles fire.

    The actual correct approach: compute XOR coherently by running BOTH
    oracles unconditionally on a single d qubit (no path ancilla needed):
      Apply Oracle_g(a) to d:   d → d ⊕ g(a)
      Apply Oracle_g(a+1) to d: d → d ⊕ g(a) ⊕ g(a+1)
    So d = g(a) ⊕ g(a+1), which is 1 iff g(a) ≠ g(a+1).

    State: (1/√(W-1)) Σ_a |a⟩ |g(a)⊕g(a+1)⟩_d

    This is NOT interference — it's just a classical XOR indicator.
    To make it interference-based we keep the path ancilla and the
    two separate g qubits, then post-recombine by checking g0 ⊕ g1 == 1.

    ── FINAL CORRECT APPROACH ────────────────────────────────────────────
    Keep two g qubits. After recombination, the p=0 branch has:
      Σ_a α_a |a⟩ (|g(a)⟩_g0|0⟩_g1 − |0⟩_g0|g(a+1)⟩_g1)

    For g(a)=g(a+1)=0: coefficient of |a⟩ = (|0⟩|0⟩ − |0⟩|0⟩) = 0  ✓ cancels
    For g(a)=g(a+1)=1: coefficient of |a⟩ = (|1⟩|0⟩ − |0⟩|1⟩) ≠ 0  ✗ survives

    So the two-g-qubit circuit correctly cancels interior-invalid regions
    (g=0,g=0) but NOT interior-valid regions (g=1,g=1).

    To also cancel interior-valid regions, we add one more step:
    5. Apply a CNOT from g0 to g1 and check if g0=g1=0 in p=0 branch.
       Actually: apply a SWAP + compare, or equivalently note that
       |1⟩_g0|0⟩_g1 − |0⟩_g0|1⟩_g1 is the |Ψ−⟩ Bell state (up to norm),
       which can be detected by H on g0, CNOT g0→g1, then measuring g1=1.

    Instead of this complexity, the cleanest implementation that achieves
    the stated goal (cancel ALL constant regions) is:
      a. Use two g qubits as above (cancels g=0,g=0 regions by interference).
      b. Add an extra CNOT-based step to handle g=1,g=1.

    For clarity of the prototype we implement the two-g-qubit version and
    note in the analysis which case cancels and which does not, so the
    user can see the interference effect clearly.
    """
    N = len(occ)
    W = N - M + 1
    assert W >= 2, "Need at least 2 anchor positions."

    g_vals = compute_g(occ, M)

    print(f"\n{'='*60}")
    print(f"Occupancy : {occ}")
    print(f"Window M  : {M},  N={N},  W={W}")
    print(f"g values  : {g_vals}  (index 0..{W-1})")
    boundaries = [a for a in range(W - 1) if g_vals[a] != g_vals[a + 1]]
    int_invalid = [a for a in range(W - 1) if g_vals[a] == 0 and g_vals[a+1] == 0]
    int_valid   = [a for a in range(W - 1) if g_vals[a] == 1 and g_vals[a+1] == 1]
    print(f"  → boundaries         (g(a)≠g(a+1)): {boundaries}")
    print(f"  → interior invalid   (g=0,g=0):      {int_invalid}  ← will cancel ✓")
    print(f"  → interior valid     (g=1,g=1):       {int_valid}  ← non-zero but orthogonal sub-components")
    print(f"{'='*60}\n")

    n_a = num_qubits_for(W)

    # ── Registers ──────────────────────────────────────────────────────────
    # TWO separate g qubits — one per branch — this is the critical fix.
    a_reg  = QuantumRegister(n_a, name='a')   # anchor
    p_reg  = QuantumRegister(1,   name='p')   # path ancilla
    g0_reg = QuantumRegister(1,   name='g0')  # g(a)   — written only in p=0 branch
    g1_reg = QuantumRegister(1,   name='g1')  # g(a+1) — written only in p=1 branch
    qc = QuantumCircuit(a_reg, p_reg, g0_reg, g1_reg)

    # ── Step 1: Uniform superposition over a = 0..W-2 ─────────────────────
    dim = 2 ** n_a
    amps = np.zeros(dim, dtype=complex)
    for a in range(W - 1):
        amps[a] = 1.0 / np.sqrt(W - 1)
    qc.initialize(amps, a_reg)
    print("Step 1 done: anchor in uniform superposition over a=0..W-2")

    # ── Step 2: Path ancilla in |-> ────────────────────────────────────────
    # |−⟩ = (|0⟩ − |1⟩)/√2  injects the minus sign between branches.
    qc.x(p_reg[0])
    qc.h(p_reg[0])
    print("Step 2 done: path ancilla in |−⟩ = (|0⟩ − |1⟩)/√2")

    # ── Step 3a: Oracle g(a) → g0_reg, controlled on p=0 ─────────────────
    # Control on p=0: sandwich with X on p.
    oracle_0 = build_g_oracle(g_vals, offset=0, n_a=n_a)
    controlled_oracle_0 = oracle_0.control(1)
    qc.x(p_reg[0])                            # flip: now fire when original p=0
    qc.append(controlled_oracle_0, [p_reg[0]] + list(a_reg) + list(g0_reg))
    qc.x(p_reg[0])                            # unflip
    print("Step 3a done: g(a)   → g0_reg  when p=0")

    # ── Step 3b: Oracle g(a+1) → g1_reg, controlled on p=1 ───────────────
    # Control on p=1: no flip needed.
    oracle_1 = build_g_oracle(g_vals, offset=1, n_a=n_a)
    controlled_oracle_1 = oracle_1.control(1)
    qc.append(controlled_oracle_1, [p_reg[0]] + list(a_reg) + list(g1_reg))
    print("Step 3b done: g(a+1) → g1_reg  when p=1")

    # ── Step 4: Recombine with Hadamard on path ancilla ───────────────────
    # After H on p, the p=0 output branch contains:
    #   Σ_a |a⟩ (|g(a)⟩_g0 |0⟩_g1  −  |0⟩_g0 |g(a+1)⟩_g1)
    # which is zero when g(a)=g(a+1)=0  (both terms = |0⟩|0⟩, opposite sign → cancel).
    qc.h(p_reg[0])
    print("Step 4 done: H on path ancilla → recombination")
    print(f"\nCircuit depth : {qc.decompose().depth()}")
    print(f"Circuit width : {qc.num_qubits} qubits\n")

    return qc, g_vals, boundaries, int_invalid, int_valid, n_a


# ─────────────────────────────────────────────────────────────────────────────
# Statevector analysis
# ─────────────────────────────────────────────────────────────────────────────

def analyse_statevector(qc: QuantumCircuit, g_vals: list[int],
                        boundaries: list[int], int_invalid: list[int],
                        int_valid: list[int], n_a: int,
                        occ: list[int], M: int):
    """
    Simulate the circuit and print amplitudes grouped by (p, a, g0, g1).

    Qubit ordering in Qiskit statevector (little-endian):
      bits 0..n_a-1  → a_reg
      bit  n_a       → p_reg[0]
      bit  n_a+1     → g0_reg[0]   (g(a),   written in p=0 branch)
      bit  n_a+2     → g1_reg[0]   (g(a+1), written in p=1 branch)

    After recombination, the p=0 branch contains:
      Σ_a |a⟩ (|g(a)⟩_g0 |0⟩_g1  −  |0⟩_g0 |g(a+1)⟩_g1)

    Cancellation:
      g(a)=g(a+1)=0 → |0⟩|0⟩ − |0⟩|0⟩ = 0         ← CANCELS ✓
      g(a)=g(a+1)=1 → |1⟩|0⟩ − |0⟩|1⟩ ≠ 0         ← does NOT cancel (orthogonal Bell-like state)
      g(a)≠g(a+1)   → boundary terms, non-zero    ← BOUNDARY ✓

    To also cancel interior-valid (g=1,g=1) regions, a further step
    is needed (e.g., Bell-basis measurement on g0,g1 or an extra CNOT
    to project onto the g0=g1=0 subspace). This is noted in the output.
    """
    W = len(g_vals)
    sv = Statevector(qc)
    amps = sv.data

    n_total = n_a + 3   # a + p + g0 + g1
    dim = 2 ** n_total

    print("\n" + "─" * 72)
    print("STATEVECTOR ANALYSIS  (p=0 branch = difference signal)")
    print("─" * 72)
    print(f"{'State |a,p,g0,g1>':26s}  {'Amplitude':>22s}  {'|Amp|²':>8s}  Note")
    print("─" * 72)

    p0_boundary_prob   = 0.0
    p0_int_invalid_prob = 0.0
    p0_int_valid_prob  = 0.0

    for idx in range(dim):
        amp = amps[idx]
        if abs(amp) < 1e-9:
            continue

        a_val  =  idx        & ((1 << n_a) - 1)
        p_val  = (idx >> n_a)       & 1
        g0_val = (idx >> (n_a + 1)) & 1
        g1_val = (idx >> (n_a + 2)) & 1
        prob   = abs(amp) ** 2

        note = ""
        if p_val == 0:                         # ← difference branch
            is_boundary    = a_val in boundaries
            is_int_invalid = a_val in int_invalid
            is_int_valid   = a_val in int_valid

            if is_boundary:
                note = "★ BOUNDARY  (survives)"
                p0_boundary_prob += prob
            elif is_int_invalid:
                note = "✓ int-invalid (g=0,g=0 — should be 0, cancel)"
                p0_int_invalid_prob += prob
            elif is_int_valid:
                note = "△ int-valid   (g=1,g=1 — Bell-like, non-zero)"
                p0_int_valid_prob += prob

        label = f"|a={a_val},p={p_val},g0={g0_val},g1={g1_val}>"
        print(f"{label:26s}  {amp.real:+.6f}{amp.imag:+.6f}j  {prob:8.5f}  {note}")

    print("─" * 72)
    print(f"\nIn the p=0 branch (difference signal):")
    print(f"  Prob on BOUNDARY anchors            : {p0_boundary_prob:.6f}  (want > 0)")
    print(f"  Prob on interior-INVALID (g=0,g=0)  : {p0_int_invalid_prob:.6f}  (want ≈ 0 ✓)")
    print(f"  Prob on interior-VALID   (g=1,g=1)  : {p0_int_valid_prob:.6f}"
          f"  (non-zero — orthogonal Bell components, needs extra step)")

    print("\n─── Boundary details ─────────────────────────────────────────────")
    for a in boundaries:
        ga, ga1 = g_vals[a], g_vals[a + 1]
        print(f"  a={a}: g({a})={ga}, g({a+1})={ga1}  "
              f"window[{a}:{a+M}]={occ[a:a+M]} → window[{a+1}:{a+1+M}]={occ[a+1:a+1+M]}")

    if int_valid:
        print("\n─── Interior-valid anchors (g=1,g=1) ────────────────────────────")
        print("  These produce |1⟩_g0|0⟩_g1 − |0⟩_g0|1⟩_g1 (|Ψ−⟩-like state).")
        print("  To suppress them: apply H⊗CNOT on (g0,g1) then discard g1=1.")
        print("  This is a one-shot projective step, NOT amplitude amplification.")
        for a in int_valid:
            print(f"  a={a}: both windows {occ[a:a+M]} and {occ[a+1:a+1+M]} are fully empty")


# ─────────────────────────────────────────────────────────────────────────────
# Step-by-step intermediate states
# ─────────────────────────────────────────────────────────────────────────────

def print_intermediate_states(occ: list[int], M: int):
    """
    Print the theoretical amplitudes at each step of the circuit.
    Explains the two-g-qubit architecture and what cancels.
    """
    N = len(occ)
    W = N - M + 1
    g_vals = compute_g(occ, M)
    W_eff = W - 1

    print("\n" + "═" * 68)
    print("THEORETICAL INTERMEDIATE STATES")
    print("═" * 68)

    print(f"\nStep 1 — uniform superposition:")
    print(f"  (1/√{W_eff}) Σ_{{a=0}}^{{{W_eff-1}}} |a⟩ |0⟩_p |0⟩_g0 |0⟩_g1")

    print(f"\nStep 2 — path ancilla → |−⟩:")
    print(f"  (1/√{W_eff})(1/√2) Σ_a |a⟩ (|0⟩−|1⟩)_p |0⟩_g0 |0⟩_g1")

    print(f"\nStep 3 — conditional oracles (g0 ← g(a) if p=0, g1 ← g(a+1) if p=1):")
    print(f"  (1/√{W_eff})(1/√2) Σ_a |a⟩ (|0⟩|g(a)⟩_g0|0⟩_g1 − |1⟩|0⟩_g0|g(a+1)⟩_g1)")
    print(f"  Per anchor:")
    for a in range(W_eff):
        ga, ga1 = g_vals[a], g_vals[a+1]
        if ga == 0 and ga1 == 0:
            fate = "→ |0⟩|0⟩ − |0⟩|0⟩ = 0        ← CANCELS after H ✓ (int-invalid)"
        elif ga == 1 and ga1 == 1:
            fate = "→ |1⟩|0⟩ − |0⟩|1⟩ ≠ 0        △ Bell-like, needs extra step (int-valid)"
        else:
            fate = f"→ |{ga}⟩|0⟩ − |0⟩|{ga1}⟩ ≠ 0  ★ BOUNDARY"
        print(f"    a={a}: g={ga}, g+1={ga1}  {fate}")

    print(f"\nStep 4 — H on p, output in p=0 branch:")
    print(f"  Σ_a |a⟩ (|g(a)⟩_g0|0⟩_g1  −  |0⟩_g0|g(a+1)⟩_g1)")
    print(f"\n  Summary:")
    print(f"    Cancelled (g=0,g=0) : {[a for a in range(W_eff) if g_vals[a]==0 and g_vals[a+1]==0]}")
    print(f"    Boundary  (g≠g+1)   : {[a for a in range(W_eff) if g_vals[a]!=g_vals[a+1]]}")
    print(f"    Bell-like (g=1,g=1) : {[a for a in range(W_eff) if g_vals[a]==1 and g_vals[a+1]==1]}")
    print("═" * 68)


def run_example(occ: list[int], M: int, label: str):
    print(f"\n{'#'*68}")
    print(f"# EXAMPLE: {label}")
    print(f"#{'='*66}#")

    print_intermediate_states(occ, M)

    result = build_difference_circuit(occ, M)
    qc, g_vals, boundaries, int_invalid, int_valid, n_a = result

    print("\nCircuit (decomposed once):")
    print(qc.decompose().draw(output='text', fold=120))

    analyse_statevector(qc, g_vals, boundaries, int_invalid, int_valid, n_a, occ, M)


# ─────────────────────────────────────────────────────────────────────────────
# 2D Generalisation (explanation)
# ─────────────────────────────────────────────────────────────────────────────

def explain_2d():
    print("""
╔══════════════════════════════════════════════════════════════╗
║           2D GENERALISATION                                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  In 2D: anchor a=(x,y), window = M×M (or M_x × M_y) block.   ║
║  g(x,y) = 1 iff the entire M×M block is zero.                ║
║                                                              ║
║  We prepare TWO difference channels in parallel:             ║
║                                                              ║
║  Channel X:  |a=(x,y)⟩ (|g(x,y)⟩ − |g(x+1,y)⟩)               ║
║  Channel Y:  |a=(x,y)⟩ (|g(x,y)⟩ − |g(x,y+1)⟩)               ║
║                                                              ║
║  Circuit structure:                                          ║
║  1. Uniform superposition over all (x,y) anchors.            ║
║  2. TWO path ancilla qubits p_x, p_y (or one with 2 levels). ║
║  3. Conditional oracles:                                     ║
║     p_x=0 → g(x,y),   p_x=1 → g(x+1,y)                       ║
║     p_y=0 → g(x,y),   p_y=1 → g(x,y+1)                       ║
║  4. Recombine with H on each path ancilla.                   ║
║                                                              ║
║  Surviving states in (p_x=1, p_y=1) branch:                  ║
║  Only (x,y) where g(x,y)≠g(x+1,y) AND g(x,y)≠g(x,y+1)        ║
║  i.e., corners of valid/invalid regions.                     ║
║                                                              ║
║  For edge detection (not just corners), measure p_x or p_y   ║
║  independently:                                              ║
║    p_x=1 branch alone → horizontal boundaries                ║
║    p_y=1 branch alone → vertical boundaries                  ║
║                                                              ║
║  The oracle for 2D g(x,y) checks M_x * M_y cells, so costs   ║
║  O(M²) gates, but is still O(1) oracle calls per anchor —    ║
║  the interference structure and query complexity are the     ║
║  same as 1D.                                                 ║
║                                                              ║
║  Full gradient vector:                                       ║
║  ∇g(a) = (g(x+1,y)-g(x,y), g(x,y+1)-g(x,y))                  ║
║  ≠ (0,0) at region boundaries, = (0,0) in interiors.         ║
╚══════════════════════════════════════════════════════════════╝
""")


# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── Example 1 ──────────────────────────────────────────────────────────
    # occ = [1, 1, 0, 0, 0, 1, 1]
    # M = 2
    # Windows of length 2:
    #   a=0: [1,1] g=0
    #   a=1: [1,0] g=0
    #   a=2: [0,0] g=1  ← valid
    #   a=3: [0,0] g=1  ← valid (interior)
    #   a=4: [0,1] g=0
    #   a=5: [1,1] g=0
    # g = [0, 0, 1, 1, 0, 0]
    # Differences (a=0..4):
    #   a=0: 0-0=0  cancel
    #   a=1: 0-1≠0  SURVIVE  (boundary: invalid→valid)
    #   a=2: 1-1=0  cancel   (interior valid)
    #   a=3: 1-0≠0  SURVIVE  (boundary: valid→invalid)
    #   a=4: 0-0=0  cancel
    # Expected surviving anchors: {1, 3}
    run_example(
        occ=[1, 1, 0, 0, 0, 1, 1],
        M=2,
        label="Block of 3 empty cells, M=2 windows"
    )

    # ── Example 2 ──────────────────────────────────────────────────────────
    # Larger example with a clear valid block interior
    # occ = [1, 0, 0, 0, 0, 0, 1]
    # M = 3
    # Windows of length 3:
    #   a=0: [1,0,0] g=0
    #   a=1: [0,0,0] g=1  ← valid
    #   a=2: [0,0,0] g=1  ← valid (interior)
    #   a=3: [0,0,0] g=1  ← valid (interior)
    #   a=4: [0,0,1] g=0
    # g = [0, 1, 1, 1, 0]
    # Differences (a=0..3):
    #   a=0: 0-1≠0  SURVIVE  (invalid→valid)
    #   a=1: 1-1=0  cancel   (interior valid)
    #   a=2: 1-1=0  cancel   (interior valid)  ← interior cancels ✓
    #   a=3: 1-0≠0  SURVIVE  (valid→invalid)
    # Expected surviving anchors: {0, 3}
    run_example(
        occ=[1, 0, 0, 0, 0, 0, 1],
        M=3,
        label="Long empty block, M=3, interior windows cancel"
    )

    # ── 2D generalisation explanation ──────────────────────────────────────
    explain_2d()
